# Libraries

In [1]:
import pandas as pd
import ast, re
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Data Inspection

In this step, all datasets used in the analysis are loaded into the notebook. These include the international job postings dataset, the **Philippine Labor Force Survey (LFS)** dataset, and **O*NET reference files** containing *occupation titles and skills*. The shapes of the datasets are printed to confirm successful loading, and a preview of the job postings dataset is displayed for initial inspection.

In [2]:
# International job postings
jobs = pd.read_csv("data_jobs.csv")

# Local LFS data
lfs = pd.read_csv("FIES2015 - LFSJAN16 CSV - Cleaned.csv", low_memory=False)

# Occupation reference files
# occupation = pd.read_excel("db_30_2_excel/Occupation Data.xlsx")
# titles = pd.read_excel("db_30_2_excel/Alternate Titles.xlsx")
# skills = pd.read_excel("db_30_2_excel/Skills.xlsx")
# tech_skills = pd.read_excel("db_30_2_excel/Technology Skills.xlsx")
# reported_titles = pd.read_excel("db_30_2_excel/Sample of Reported Titles.xlsx")

print(jobs.shape)
print(lfs.shape)

jobs.head()

(785741, 17)
(207212, 154)


,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills
0,Senior Data Engineer,Senior Clinical Data Engineer / Principal Clin...,"Watertown, CT",via Work Nearby,Full-time,False,"Texas, United States",2023-06-16 13:44:15,False,False,United States,NaN,NaN,NaN,Boehringer Ingelheim,NaN,NaN
1,Data Analyst,Data Analyst,"Guadalajara, Jalisco, Mexico",via BeBee México,Full-time,False,Mexico,2023-01-14 13:18:07,False,False,Mexico,NaN,NaN,NaN,Hewlett Packard Enterprise,"['r', 'python', 'sql', 'nosql', 'power bi', 't...","{'analyst_tools': ['power bi', 'tableau'], 'pr..."
2,Data Engineer,"Data Engineer/Scientist/Analyst, Mid or Senior...","Berlin, Germany",via LinkedIn,Full-time,False,Germany,2023-10-10 13:14:55,False,False,Germany,NaN,NaN,NaN,ALPHA Augmented Services,"['python', 'sql', 'c#', 'azure', 'airflow', 'd...","{'analyst_tools': ['dax'], 'cloud': ['azure'],..."
3,Data Engineer,LEAD ENGINEER - PRINCIPAL ANALYST - PRINCIPAL ...,"San Antonio, TX",via Diversity.com,Full-time,False,"Texas, United States",2023-07-04 13:01:41,True,False,United States,NaN,NaN,NaN,Southwest Research Institute,"['python', 'c++', 'java', 'matlab', 'aws', 'te...","{'cloud': ['aws'], 'libraries': ['tensorflow',..."
4,Data Engineer,Data Engineer- Sr Jobs,"Washington, DC",via Clearance Jobs,Full-time,False,Sudan,2023-08-07 14:29:36,False,False,Sudan,NaN,NaN,NaN,Kristina Daniel,"['bash', 'python', 'oracle', 'aws', 'ansible',...","{'cloud': ['oracle', 'aws'], 'other': ['ansibl..."


In [3]:
jobs.columns

Index(['job_title_short', 'job_title', 'job_location', 'job_via',
       'job_schedule_type', 'job_work_from_home', 'search_location',
       'job_posted_date', 'job_no_degree_mention', 'job_health_insurance',
       'job_country', 'salary_rate', 'salary_year_avg', 'salary_hour_avg',
       'company_name', 'job_skills', 'job_type_skills'],
      dtype='object')

In [4]:
lfs.columns

Index(['id', 'person', 'w_regn', 'other_id', 'rfact', 'pop_adj', 'urb', 'tstr',
       'tpsu', 'pcinc',
       ...
       'high_allobs', 'low_allobs', 'high_allobs_emp', 'low_allobs_emp',
       'psic', 'industrygrp1', 'industrygrp2', 'educgrp1', 'classworkrgrp',
       'agegrp1'],
      dtype='object', length=154)

# EDA

# Preprocessing the International Dataset

This step creates a working dataset from the international job postings data by selecting only the columns needed for the analysis. The selected columns include job title, country, salary information, and listed skills. A copy of the filtered dataset is created to avoid modifying the original data.

In [5]:
jobs_model = jobs[[
    "job_title_short", "job_country",
    "salary_rate", "salary_year_avg", "salary_hour_avg",
    "job_skills"
]].copy()

## Standardizing the Salary

This step standardizes salary values by creating a single yearly salary column. If a job posting already contains an average yearly salary (`salary_year_avg`), **that value is used directly**. If only an hourly salary (`salary_hour_avg`) is available, it is **converted to a yearly salary** by *multiplying it by 40 hours per week and 52 weeks per year*. Rows without salary information are removed to ensure consistent salary values for analysis.

*Note:* The conversion assumes a standard full-time schedule of 40 hours per week and 52 weeks per year.

**Source**
- https://clockify.me/blog/managing-time/how-many-work-hours-in-a-year/

In [6]:
def get_salary(row):
    if pd.notna(row["salary_year_avg"]):
        return row["salary_year_avg"]
    if pd.notna(row["salary_hour_avg"]):
        return row["salary_hour_avg"] * 40 * 52
    return None

jobs_model["salary"] = jobs_model.apply(get_salary, axis=1)
jobs_model = jobs_model.dropna(subset=["salary"])

In [7]:
# Remove salary outliers (IQR method)
Q1 = jobs_model["salary"].quantile(0.25)
Q3 = jobs_model["salary"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

jobs_model = jobs_model[
    (jobs_model["salary"] >= lower) & 
    (jobs_model["salary"] <= upper)
]

## Preprocessing the Job Roles

*Note:* No filtering for data industry roles was required because the dataset already consists of job postings related to data and analytics roles. According to the dataset description, it contains real-world data analytics job postings collected from various online job sources. Therefore, all job titles are assumed to belong to the data industry. A count of job titles was performed to examine the distribution of roles in the dataset.

**Source**
- https://huggingface.co/datasets/lukebarousse/data_jobs

In [8]:
jobs_model["job_title_short"].value_counts()

job_title_short
Data Analyst                 9566
Data Scientist               8289
Data Engineer                6681
Senior Data Engineer         1980
Senior Data Scientist        1942
Senior Data Analyst          1476
Business Analyst              996
Machine Learning Engineer     610
Software Engineer             561
Cloud Engineer                 85
Name: count, dtype: int64

In [9]:
title_counts = jobs_model["job_title_short"].value_counts()

common_titles = title_counts[title_counts >= 100].index

jobs_model = jobs_model[jobs_model["job_title_short"].isin(common_titles)]

This **function** cleans and standardizes the skill data by converting different formats (lists, strings, or missing values) into a consistent list of individual skills. It removes extra characters, splits merged skills, and ensures each job posting returns a clean list of skills.

In [10]:
def clean_skills(x):
    if isinstance(x, list):
        all_items = []
        for item in x:
            text = str(item).strip().strip("'\"[]")
            parts = re.split(r",\s*", text)
            all_items.extend(parts)
        return [s.strip().strip("'\" ").lower() for s in all_items if s.strip()]
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return clean_skills(parsed)
        except:
            pass
        x_clean = x.strip().strip("[]")
        parts = re.split(r",\s*", x_clean)
        return [s.strip().strip("'\" ").lower() for s in parts if s.strip()]
    return []

Applying the `clean_skills` function to the `job_skills` column to create a clean and standardized list of skills for each job posting.

In [11]:
jobs_model["skills_list"] = jobs_model["job_skills"].apply(clean_skills)

This step removes duplicate skills within each row while preserving the original order of skills.

In [12]:
jobs_model["skills_list"] = jobs_model["skills_list"].apply(lambda x: list(dict.fromkeys(x)))

In [13]:
jobs_model["skills_list"] = jobs_model["skills_list"].apply(lambda x: x if isinstance(x, list) else [])

This step ensures that all values in `skills_list` are valid lists to avoid errors in later processing.

In [14]:
jobs_model["skill_count"] = jobs_model["skills_list"].apply(len)

This dictionary maps countries to broader geographic regions (e.g., NA, EU, SEA). It allows the dataset to group job postings by region for easier comparison and analysis.

In [15]:
region_map = {

    # North America
    "United States": "NA",
    "Canada": "NA",

    # Latin America
    "Mexico": "LATAM",
    "Costa Rica": "LATAM",
    "Honduras": "LATAM",
    "Guatemala": "LATAM",
    "Panama": "LATAM",
    "Bahamas": "LATAM",
    "Jamaica": "LATAM",
    "Puerto Rico": "LATAM",
    "Dominican Republic": "LATAM",
    "Colombia": "LATAM",
    "Brazil": "LATAM",
    "Argentina": "LATAM",
    "Chile": "LATAM",
    "Peru": "LATAM",
    "Uruguay": "LATAM",
    "Bolivia": "LATAM",
    "Paraguay": "LATAM",
    "Venezuela": "LATAM",
    "El Salvador": "LATAM",
    "Nicaragua": "LATAM",

    # Europe
    "United Kingdom": "EU",
    "Germany": "EU",
    "France": "EU",
    "Spain": "EU",
    "Belgium": "EU",
    "Denmark": "EU",
    "Hungary": "EU",
    "Poland": "EU",
    "Portugal": "EU",
    "Sweden": "EU",
    "Estonia": "EU",
    "Croatia": "EU",
    "Netherlands": "EU",
    "Ireland": "EU",
    "Lithuania": "EU",
    "Slovenia": "EU",
    "Austria": "EU",
    "Greece": "EU",
    "Czechia": "EU",
    "Switzerland": "EU",
    "Italy": "EU",
    "Norway": "EU",
    "Luxembourg": "EU",
    "Romania": "EU",
    "Finland": "EU",
    "Latvia": "EU",
    "Slovakia": "EU",
    "Serbia": "EU",
    "Montenegro": "EU",
    "Bosnia and Herzegovina": "EU",
    "Belarus": "EU",
    "Ukraine": "EU",

    # Southeast Asia
    "Philippines": "SEA",
    "Singapore": "SEA",
    "Indonesia": "SEA",
    "Malaysia": "SEA",
    "Thailand": "SEA",
    "Vietnam": "SEA",
    "Cambodia": "SEA",
    "Myanmar": "SEA",
    "Brunei": "SEA",

    # East Asia
    "Japan": "EAST_ASIA",
    "South Korea": "EAST_ASIA",
    "Taiwan": "EAST_ASIA",
    "China": "EAST_ASIA",
    "Hong Kong": "EAST_ASIA",

    # South Asia
    "India": "SOUTH_ASIA",
    "Pakistan": "SOUTH_ASIA",
    "Bangladesh": "SOUTH_ASIA",
    "Sri Lanka": "SOUTH_ASIA",
    "Nepal": "SOUTH_ASIA",

    # Middle East
    "Israel": "ME",
    "United Arab Emirates": "ME",
    "Jordan": "ME",
    "Turkey": "ME",
    "Lebanon": "ME",

    # Africa
    "South Africa": "AFRICA",
    "Nigeria": "AFRICA",
    "Kenya": "AFRICA",
    "Ghana": "AFRICA",
    "Morocco": "AFRICA",
    "Egypt": "AFRICA",
    "Zimbabwe": "AFRICA",
    "Namibia": "AFRICA",
    "Tunisia": "AFRICA",
    "Algeria": "AFRICA",
    "Uganda": "AFRICA",
    "Cameroon": "AFRICA",
    "Senegal": "AFRICA",
    "Mali": "AFRICA",
    "Benin": "AFRICA",
    "Sudan": "AFRICA",
    "Zambia": "AFRICA",

    # Oceania
    "Australia": "OCEANIA",
    "New Zealand": "OCEANIA"
}

This step creates a new `region` column by mapping each country in `job_country` to its corresponding geographic region using the `region_map` dictionary. Countries that are not included in the mapping are assigned the value `"OTHER"` to handle unmatched cases.

In [16]:
jobs_model["region"] = jobs_model["job_country"].map(region_map).fillna("OTHER")

This step converts the list of skills into numerical features using multi-hot encoding. Each skill becomes a column, where 1 indicates the skill is present and 0 indicates it is absent.

In [17]:
mlb = MultiLabelBinarizer()

skills_matrix = mlb.fit_transform(jobs_model["skills_list"])

skills_df = pd.DataFrame(skills_matrix, columns=mlb.classes_, index=jobs_model.index)

**Skill Frequency Inspection**

In this step, the frequency of each skill in the dataset is examined. Because the skills have been multi-hot encoded, summing each column counts how many job postings contain that skill. This allows us to observe the overall distribution of skills across the dataset.

The results show that some skills appear very frequently (such as SQL, Python, Tableau, and R), while many others appear only a few times. A large number of these skills occur in very small counts (for example, appearing only once or a handful of times). These **rare skills** can introduce noise and unnecessarily increase the number of features in the model.

To improve model stability and reduce dimensionality, the next step filters out these rare skills and keeps only those that appear above a minimum frequency threshold.

In [18]:
skill_counts = skills_df.sum().sort_values(ascending=False)

top_skills = skill_counts.head(150).index

skills_df = skills_df[top_skills]

This step removes rarely occurring skills from the dataset. The frequency of each skill is calculated by summing the multi-hot encoded skill columns. Skills that appear fewer than 50 times are considered rare and are removed, while only commonly occurring skills are retained. This reduces noise and limits the number of features used in the model.

Before adding the filtered set of skill features back into the dataset, the original skill columns are removed from `jobs_model`. The list `mlb.classes_` contains all skills generated during the initial multi-hot encoding step, including rare skills that were later filtered out. Dropping these columns ensures that the modeling dataset only includes the final filtered skill features.

In [19]:
# Drop old skill columns
jobs_model = jobs_model.drop(columns=mlb.classes_, errors="ignore")

## Training the Model

After filtering the skills to keep only commonly occurring ones, the modeling dataset is rebuilt by combining the filtered skill features with the existing job information in `jobs_model`. This ensures that the dataset used for modeling includes only the selected skill features.

In [20]:
jobs_model = pd.concat([jobs_model, skills_df], axis=1)

This step converts the categorical `region` variable into numerical features using one-hot encoding. Each region becomes a separate binary column indicating whether a job posting belongs to that region. These encoded region variables are then added to the modeling dataset so that geographic location can be used as a feature in the model.

This step converts the categorical `job_title_short` variable into numerical features using one-hot encoding. Each job title is transformed into a separate binary column indicating whether a job posting belongs to that role. These encoded job title features are then added to the modeling dataset so that the model can capture differences in salary across job roles.

In [21]:
region_dummies = pd.get_dummies(jobs_model["region"], prefix="region")

title_dummies = pd.get_dummies(jobs_model["job_title_short"], prefix="title")

jobs_model = pd.concat([jobs_model, region_dummies, title_dummies], axis=1)

In [22]:
jobs_model["cloud_skills"] = jobs_model[["aws","azure","gcp"]].sum(axis=1)

jobs_model["bi_tools"] = jobs_model[["tableau","power bi","looker","qlik"]].sum(axis=1)

jobs_model["ml_tools"] = jobs_model[["tensorflow","pytorch","scikit-learn","keras"]].sum(axis=1)

jobs_model["is_US"] = (jobs_model["job_country"] == "United States").astype(int)

jobs_model["salary_log"] = np.log1p(jobs_model["salary"])

In [23]:
X = jobs_model.drop(columns=[
    "job_title_short","job_country",
    "salary_rate","salary_year_avg","salary_hour_avg",
    "job_skills","skills_list","region",
    "salary","salary_log"
])

The target variable `y`,

In [24]:
y = jobs_model["salary_log"]

This step divides the dataset into training and testing sets using `train_test_split`. The feature matrix `X` and target variable `y` are split so that 80% of the data is used to train the model and 20% is reserved for testing. The `random_state=42` parameter ensures that the split is reproducible, meaning the same data partition will be generated each time the code is run.

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [27]:
lr = LinearRegression()
rf = RandomForestRegressor(random_state=42)
gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=5, random_state=42)
xgb = XGBRegressor(n_estimators=700, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)

In [28]:
param_grid = {"n_estimators": [300,500,700], "max_depth":[10,15,20,None], "min_samples_split":[5,10,20]}
search = RandomizedSearchCV(rf, param_distributions=param_grid, n_iter=10, cv=3, scoring="r2", n_jobs=-1, random_state=42)
search.fit(X_train, y_train)
rf_best = search.best_estimator_
print("Best RF parameters:", search.best_params_)

Best RF parameters: {'n_estimators': 700, 'min_samples_split': 20, 'max_depth': None}


Initializing the models,

Training the models,

In [29]:
lr.fit(X_train_scaled, y_train)
rf_best.fit(X_train, y_train)
gb.fit(X_train, y_train)
xgb.fit(X_train, y_train)

print("All models trained successfully!")

All models trained successfully!


Making predictions,

In [30]:
lr_pred = np.expm1(lr.predict(X_test_scaled))
rf_pred = np.expm1(rf_best.predict(X_test))
gb_pred = np.expm1(gb.predict(X_test))
xgb_pred = np.expm1(xgb.predict(X_test))
y_test_salary = np.expm1(y_test)

Evaluating the models,

In [31]:
results = pd.DataFrame({
    "Model": ["Linear Regression","Random Forest","Gradient Boosting","XGBoost"],
    "R2": [
        r2_score(y_test, np.log1p(lr_pred)),
        r2_score(y_test, np.log1p(rf_pred)),
        r2_score(y_test, np.log1p(gb_pred)),
        r2_score(y_test, np.log1p(xgb_pred))
    ],
    "MAE": [
        mean_absolute_error(y_test_salary, lr_pred),
        mean_absolute_error(y_test_salary, rf_pred),
        mean_absolute_error(y_test_salary, gb_pred),
        mean_absolute_error(y_test_salary, xgb_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test_salary, lr_pred)),
        np.sqrt(mean_squared_error(y_test_salary, rf_pred)),
        np.sqrt(mean_squared_error(y_test_salary, gb_pred)),
        np.sqrt(mean_squared_error(y_test_salary, xgb_pred))
    ]
})
print(results)

               Model        R2           MAE          RMSE
0  Linear Regression  0.249919  29806.107725  38014.383136
1      Random Forest  0.304118  27934.951994  36328.443595
2  Gradient Boosting  0.284420  29010.111244  37166.032221
3            XGBoost  0.302674  28411.585170  36593.822650


In [32]:
cv_scores = cross_val_score(rf_best, X, y, cv=5, scoring="r2")
print("RF CV mean R2:", cv_scores.mean())

RF CV mean R2: 0.2810756502006206
